In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "cat_ejec_dbricks")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "saejercdatabricks")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/Suspensiones.csv"

In [0]:
df_suspens = (
    spark.read.option("header", True)\
    #.schema(suspens_schema)\
    .option("inferSchema", True)
    .option("sep", ";")
    .option("multiLine", True) 
    .option("quote", '"')
    .option("escape", '"') 
    .option("encoding", "ISO-8859-1") 
    .csv(ruta)
)

In [0]:
from pyspark.sql import functions as F

columnas_fecha = [
    "FECHA_GRABADA",
    "FECHA_REALIZADA",
    "FECREA_FIN",
]

for col_name in columnas_fecha:
    col_limpia = F.regexp_replace(
        F.regexp_replace(F.col(col_name), r"[\?\.\u00A0\u202F]", ""),
        r"(?i)(am|pm)$",
        " $1",
    )
    df_suspens = df_suspens.withColumn(
        col_name,
        F.coalesce(
            F.try_to_timestamp(col_limpia, F.lit("dd/MM/yyyy h:mm:ss a")),
            F.try_to_timestamp(F.col(col_name), F.lit("yyyy-MM-dd HH:mm:ss")),
        ),
    )

In [0]:
df_suspens_selected = df_suspens.select(col('codigo').alias('OT_SUSPENSION'), 
                                                   col('descripcion'), 
                                                   col('observacion'),
                                                   col('FECHA_GRABADA').alias('FECHA GRABADA'),
                                                   col('FECHA_REALIZADA').alias('FECHA REALIZADA'),
                                                   col('FECREA_FIN').alias('FECHA FINALIZACION'), 
                                                   col('grupo'),
                                                   col('suscriptor'),
                                                   col('ciclo'),
                                                   col('estrato'),
                                                   col('finalizada_por').alias('FUNCIONARIO_REALIZA'),
                                                   col('deuda')
                                                   )

In [0]:
df_suspens_final = df_suspens_selected.withColumn("INGESTION_DATE", current_timestamp())

In [0]:
df_suspens_final.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.suspensiones")